# 01 · Build the landmark dataset

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsemaniJoshua/SignLens/blob/main/notebooks/01_build_landmark_dataset.ipynb)

Collects hand landmarks from every available source into two files in your Google Drive:

* `SignLens/data/static_landmarks.npz`: one hand per sample, for letters held still (notebook 02)
* `SignLens/data/motion_sequences.npz`: short clips, for signs that move, J and Z (notebook 03)

**Sources** (switch each on or off in the next cell):

| source | what it adds | needs |
|---|---|---|
| repo data | YouTube signer (A–Z) + 30 webcam recordings per letter A–S + digit videos | nothing |
| Kaggle datasets | thousands of photos from other people, cameras and backgrounds | free Kaggle account (see below) |
| your photos | `MyDrive/SignLens/my_photos/<LETTER>/*.jpg` | photos in Drive |
| your videos | `MyDrive/SignLens/my_videos/<SIGN>/*.mp4`. Short phone clips work well, and **friends and family count**: every new person helps more than anything else | videos in Drive |

**Why "recordings" matter more than "images":** 300 frames from one 10-second clip is roughly 1 example. 30 short clips from 10 different people is a real dataset. The table at the end shows recordings per class.

> **Kaggle setup (once):** kaggle.com → Settings → *Create New Token*. In Colab, click the 🔑 *Secrets* icon on the left and add `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access switched on.

> **ASL vs GSL:** the public datasets and the YouTube video are **ASL**. GSL is strongly ASL-influenced, but confirm the GSL handshapes with a GSL signer or dictionary and add GSL recordings before describing the model as GSL in a publication.

In [ ]:
#@title Setup: get the code and data (run this first)
import os, sys, subprocess

REPO_URL = "https://github.com/AsemaniJoshua/SignLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    REPO_DIR = "/content/SignLens"
    if not os.path.isdir(REPO_DIR):
        # Shallow + sparse clone: only notebooks/ and the small keypoint files
        # (~225 MB, a few seconds). Skips node_modules, videos and images.
        sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {BRANCH} "
           f"--filter=blob:none --sparse {REPO_URL} {REPO_DIR}")
        sh(f"cd {REPO_DIR} && git sparse-checkout set --no-cone /notebooks/ "
           "/model/dataset/static_keypoints_optimized/ "
           "'/model/dataset/video_keypoints_no_face/*/[0-9].npy' "
           "'/model/initial build/dataset/' /model/model_endpoints/app/")
    sh("pip install -q mediapipe kagglehub")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SignLens"   # outputs survive the session
else:
    REPO_DIR = os.environ.get("SIGNLENS_REPO_DIR", os.path.abspath(".."))
    WORK_DIR = os.environ.get("SIGNLENS_WORK_DIR", os.path.join(REPO_DIR, "work"))

MODEL_DIR = os.path.join(REPO_DIR, "model")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR = os.path.join(WORK_DIR, "models")
for d in (WORK_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))

import numpy as np
import signlens_core as sc
QUICK = os.environ.get("SIGNLENS_QUICK") == "1"   # tiny settings for automated tests
print(f"signlens_core {sc.__version__} | repo data: {MODEL_DIR} | outputs: {WORK_DIR}")

In [ ]:
#@title Options
USE_REPO_DATA = True  #@param {type:"boolean"}
USE_KAGGLE = True  #@param {type:"boolean"}
KAGGLE_DATASETS = {
    # slug: (use it?, files per "recording" bucket, note)
    "grassknoted/asl-alphabet": (True, 50, "87k photos, mostly 1 signer, plain background"),
    "danrasband/asl-alphabet-test": (True, 5, "small, several people and real backgrounds; great TEST source"),
    "lexset/synthetic-asl-alphabet": (True, 1, "synthetic 3D hands: many skin tones, angles, lighting"),
    "debashishsau/aslamerican-sign-language-aplhabet-dataset": (False, 50, "large; switch on for more variety"),
}
MAX_PER_CLASS = 300  #@param {type:"integer"}
MY_PHOTOS_DIR = os.path.join(WORK_DIR, "my_photos")
MY_VIDEOS_DIR = os.path.join(WORK_DIR, "my_videos")
MOTION_LABELS = ["J", "Z"]   # signs defined by movement -> motion model
HAND_MODEL = os.path.join(WORK_DIR, "hand_landmarker.task")
CACHE_DIR = os.path.join(DATA_DIR, "cache")      # extracted landmarks per source (re-runs are instant)
os.makedirs(CACHE_DIR, exist_ok=True)
if QUICK:
    USE_KAGGLE = False
static_parts, seq_parts = [], []   # filled by the cells below

## 1. Data already in the repo

In [ ]:
if USE_REPO_DATA:
    yt = sc.load_repo_youtube_frames(MODEL_DIR)
    cam = sc.load_repo_webcam_sequences(MODEL_DIR, frame_step=2)
    have = set(yt.label) | set(cam.label)
    vidframes = sc.load_repo_video_frames(MODEL_DIR, labels=None)
    vidframes = vidframes.subset(~np.isin(vidframes.label, list(have) + MOTION_LABELS))  # digits only
    static_parts += [yt, cam, vidframes]
    for s, lab, g in sc.load_repo_video_sequences(MODEL_DIR):
        if lab in MOTION_LABELS:
            seq_parts.append((s, lab, g, "repo_video"))
    print(f"repo: {len(yt)} YouTube + {len(cam)} webcam + {len(vidframes)} digit-video frames, "
          f"{len(seq_parts)} motion sequences")

## 2. MediaPipe hand model

The old code used `mediapipe.solutions`, which current MediaPipe no longer includes. The new code uses the Tasks `HandLandmarker` with this model file (downloaded once to Drive).

In [ ]:
from signlens_core import extract
need_mediapipe = USE_KAGGLE or os.path.isdir(MY_PHOTOS_DIR) or os.path.isdir(MY_VIDEOS_DIR)
if need_mediapipe:
    extract.download_model(HAND_MODEL)
    print("hand model ready:", HAND_MODEL)
else:
    print("no new images/videos to process; skipping MediaPipe")

## 3. Public datasets from Kaggle

Each dataset becomes its own **source**, so notebook 02 can test on a dataset the model never saw. Extraction runs once and is cached in Drive (roughly 2–5 minutes per dataset at 300 images per letter).

In [ ]:
def kaggle_login():
    try:
        from google.colab import userdata
        for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
            os.environ.setdefault(k, userdata.get(k))
    except Exception as e:
        print("Kaggle secrets not found in Colab (", e, "). Trying anonymous download.")

if USE_KAGGLE:
    import kagglehub
    kaggle_login()
    for slug, (use, bucket, note) in KAGGLE_DATASETS.items():
        if not use:
            continue
        src = slug.split("/")[1][:24]
        cache = os.path.join(CACHE_DIR, f"kaggle_{src}_{MAX_PER_CLASS}.npz")
        if os.path.exists(cache):
            part = sc.StaticSet.load(cache)
            print(f"{slug}: {len(part)} samples (cached)")
        else:
            print(f"downloading {slug}  ({note})")
            try:
                path = kagglehub.dataset_download(slug)
                part = extract.extract_image_dataset(path, HAND_MODEL, source=src,
                                                     max_per_class=MAX_PER_CLASS, bucket=bucket)
                part.save(cache)
            except Exception as e:
                print(f"  !! skipped {slug}: {e}")
                continue
        static_parts.append(part.subset(~np.isin(part.label, MOTION_LABELS)))

## 4. Your own photos and videos

Put files in Google Drive like this (folder names are the sign):

```
MyDrive/SignLens/my_photos/A/IMG_001.jpg ...
MyDrive/SignLens/my_videos/A/ama_01.mp4      <- 3-5 s holding A, hand moving slightly
MyDrive/SignLens/my_videos/J/kofi_01.mp4     <- one J per clip
```

Tips that matter: different people, different rooms and lighting, the camera at roughly chest height, the whole hand in the frame. Put the person's name in the file name (`ama_01.mp4`) so you can later test on a person the model never saw.

In [ ]:
def person_of(group):            # "my_videos:A/ama_01.mp4" -> "ama"
    base = group.split("/")[-1]
    return base.split("_")[0].lower() if "_" in base else "me"

if os.path.isdir(MY_PHOTOS_DIR):
    part = extract.extract_image_dataset(MY_PHOTOS_DIR, HAND_MODEL, source="my_photos",
                                         max_per_class=10_000, bucket=10)
    static_parts.append(part)
if os.path.isdir(MY_VIDEOS_DIR):
    stat, seqs, labs, grps = extract.extract_video_dataset(MY_VIDEOS_DIR, HAND_MODEL, MOTION_LABELS)
    if len(stat):
        # one source per person -> notebook 02 can test on an unseen PERSON
        stat.source = np.array([f"person:{person_of(g)}" for g in stat.group], dtype=object)
        static_parts.append(stat)
    seq_parts += [(s, l, g, f"person:{person_of(g)}") for s, l, g in zip(seqs, labs, grps)]
if not (os.path.isdir(MY_PHOTOS_DIR) or os.path.isdir(MY_VIDEOS_DIR)):
    for d in (MY_PHOTOS_DIR, MY_VIDEOS_DIR):
        print("not found (optional):", d)

## 5. Combine, check and save

In [ ]:
static = sc.StaticSet.concat(static_parts)
static = static.subset(~np.isin(static.label, MOTION_LABELS))
static.save(os.path.join(DATA_DIR, "static_landmarks.npz"))
static.summary()

if seq_parts:
    s, l, g, src = zip(*seq_parts)
    sc.save_sequences(os.path.join(DATA_DIR, "motion_sequences.npz"), s, l, g, src)
from collections import Counter
seq_recs = Counter(l for _, l, _, _ in seq_parts)
print("\nmotion sequences:", dict(seq_recs) or "none")
print("saved to", DATA_DIR)

In [ ]:
#@title What to collect next
recs = Counter((l, g) for l, g in zip(static.label, static.group))
per_class = Counter(l for l, _ in recs)
sources_per_class = {l: len(set(static.source[static.label == l])) for l in set(static.label)}
print("Static letters: distinct recordings (sources) per class")
for lab in sc.sort_labels(set(sc.data.LETTERS) - set(MOTION_LABELS) | set(static.label)):
    n, k = per_class.get(lab, 0), sources_per_class.get(lab, 0)
    flag = "  <- needs data" if n < 20 or k < 3 else ""
    print(f"  {lab:3s} {n:5d} rec from {k} sources{flag}")
print("\nMotion signs (need ~20+ clips from several people each):")
for lab in MOTION_LABELS:
    n = seq_recs.get(lab, 0)
    print(f"  {lab}: {n} clips" + ("  <- record more" if n < 20 else ""))

Next: **02_train_static_model.ipynb** (letters) and **03_train_motion_model.ipynb** (J, Z).